# VideoDB Understanding: Activity

Turn a video into a timeline of actions and goals with the `activity` analyzer. Choose Gemini or an Open Compute model, then inspect the actions and play the corresponding video moments.

This example uses a first-person video of a person folding laundry. Activity handles frame preparation and both analysis passes for you.

<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/preview/guides/indexing-v2/understanding/activity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Install dependencies

In [ ]:
%pip install -q "videodb>=0.5.0" python-dotenv pandas yt-dlp

## 2. Connect to VideoDB

Set `VIDEO_DB_BASE_URL` in your environment or `.env` to your VideoDB endpoint with Activity support deployed. Use your VideoDB API key when prompted.

For Gemini, your deployment needs Gemini access. For Open Compute, have an active sandbox with a model that supports both vision and text generation.

In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv
from videodb import connect

load_dotenv()
api_key = os.getenv("VIDEO_DB_API_KEY") or getpass("Enter your VideoDB API key: ")

conn = connect(api_key=api_key, base_url=os.environ["VIDEO_DB_BASE_URL"])
collection = conn.get_collection()

## 3. Choose a video

Download the sample video locally with `yt-dlp` and upload the file. Replace the URL with your own video, or reuse an uploaded video. Merging the video and audio streams needs `ffmpeg`, which Colab already has.

In [ ]:
!yt-dlp -q -f "bv*[ext=mp4][height<=1080]+ba[ext=m4a]/b[ext=mp4]/b" --merge-output-format mp4 -o laundry-folding-pov.mp4 "https://www.youtube.com/watch?v=wuRf25_zME0"

In [ ]:
video = collection.upload(file_path="laundry-folding-pov.mp4")

# Alternatively, skip the download cell and replace the file upload above with:
# video = collection.upload(url="https://www.youtube.com/watch?v=wuRf25_zME0")

# Or reuse an uploaded video:
# video = collection.get_video("YOUR_VIDEO_ID")

video.play()

## 4. Choose a provider

**Run one of the next two submission cells.** Each cell creates a new Understanding run. The remaining cells work with either provider.

### Option A: Gemini

- **`express`** calls the Gemini API directly.
- **`batch`** submits Gemini batch jobs, which can take longer. This is the default.

Both modes return an Understanding ID immediately; wait for the results below. Use a Gemini model available to your deployment. The API accepts Gemini model names; an accepted name does not guarantee provider availability.

Activity is billed under custom LLM usage: the calculated Gemini token cost across both passes, including repair requests, plus 20%. For example, a $2.40 provider cost produces a $2.88 charge. Unknown rates or missing usage remain pending rather than becoming a zero-cost charge.

Gemini batch input and output token rates are 50% lower than express rates. Cache reads use the published rate for the selected model. We apply the 20% markup after calculating the discounted provider cost.

In [ ]:
GEMINI_MODEL = "google/gemini-2.5-pro"
GEMINI_MODE = "batch"  # @param ["express", "batch"]

run = video.understand(analyzers=[{
    "type": "activity",
    "name": "activity",
    "config": {"model": GEMINI_MODEL, "mode": GEMINI_MODE},
}])
print("Understanding ID:", run.id)

### Option B: Open Compute

This example uses `Qwen/Qwen3.5-9B`. Open Compute handles inference as jobs; **omit `mode` entirely**. Sending `mode`, including `batch`, is rejected.

Supply your sandbox ID below, or leave it empty to select a compatible active sandbox automatically. The model must be available and enabled for both vision and text generation. Open Compute uses sandbox runtime billing, without the Gemini token markup.

In [ ]:
OPEN_COMPUTE_MODEL = "Qwen/Qwen3.5-9B"
SANDBOX_ID = ""  # Optional: your active sandbox ID.

activity_config = {"model": OPEN_COMPUTE_MODEL}
if SANDBOX_ID:
    activity_config["sandbox_id"] = SANDBOX_ID

run = video.understand(analyzers=[{
    "type": "activity",
    "name": "activity",
    "config": activity_config,
}])
print("Understanding ID:", run.id)

## 5. Wait for the results

Batch jobs can take longer than express requests. If this cell times out, rerun **this cell** to keep waiting for the same Understanding ID. Rerunning a submission cell creates another run.

In [ ]:
run.wait_until_complete(timeout=48 * 3600, poll_interval=60)
analyzer = run.get_analyzer("activity", refresh=True)
print("Status:", analyzer.status)
if not analyzer.is_successful:
    raise RuntimeError(f"Activity did not succeed: {analyzer.status}")

activity_output = analyzer.get_output()

## 6. Read the action timeline

`scenes` contains the actions. Each entry has `scene_id`, `start` and `end` in seconds, plus `data` describing the action, its goal, confidence and any issue. `metadata.goals` groups actions into higher-level goals.

Warnings matter: a completed run may contain uncertain or unavailable intervals. Review warnings and rows with an issue before relying on the timeline.

In [ ]:
import pandas as pd
from IPython.display import display

scenes = activity_output.get("scenes", [])
metadata = activity_output.get("metadata", {})
if metadata.get("warnings"):
    print("Processing warnings:", metadata["warnings"])

rows = []
for scene in scenes:
    data = scene.get("data") or {}
    rows.append({
        "Scene ID": scene["scene_id"],
        "Start (s)": scene["start"],
        "End (s)": scene["end"],
        "Action": data.get("instruction"),
        "Goal": data.get("goal_instruction"),
        "Confidence": data.get("confidence"),
        "Issue": data.get("issue"),
    })
actions = pd.DataFrame(rows)
display(actions.head(20))

### Goals

A goal can span several actions. An unavailable start or end means the output did not contain enough supporting actions to establish that goal's interval.

In [ ]:
goals = pd.DataFrame(metadata.get("goals", []))
goal_columns = [column for column in ["goal_id", "instruction", "start", "end", "uncertainty"] if column in goals]
display(goals[goal_columns])

## 7. Play an action

Choose a row from the action table to generate a stream for just that moment. The stream URL can also be used in your own player. Generating a stream does not rerun Activity.

In [ ]:
from videodb import play_stream

ACTION_INDEX = 0  # Change this to another row number.
if not scenes:
    raise ValueError("No action intervals were returned.")
scene = scenes[ACTION_INDEX]
start, end = scene.get("start"), scene.get("end")
if start is None or end is None or end <= start:
    raise ValueError("Choose an action with a valid time interval.")

stream_url = video.generate_stream(timeline=[(start, end)])
print(f"{scene['scene_id']}: {start:.2f}s–{end:.2f}s")
print("Stream URL:", stream_url)
play_stream(stream_url)

### Inspect usage

Gemini results include the provider cost, 20% markup and billable cost when all request usage can be priced. Token totals are also available per pass. Open Compute results do not include a Gemini token charge.

In [ ]:
usage_summary = {key: metadata[key] for key in [
    "model", "mode", "cost_status", "estimated_cost_usd", "markup_percent", "billable_cost_usd"
] if key in metadata}
display(pd.DataFrame([usage_summary]))